In [ ]:
# Thử nghiệm ViT5 tóm tắt tin tức tiếng Việt
import torch
%pip install -q transformers sentencepiece accelerate
import transformers, sentencepiece, accelerate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print(torch.__version__, torch.cuda.is_available())
model_name = "NishiKyen/vit5-vietnamese-news"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print("Thiết bị:", device)

# Dán văn bản bài ngắn để thử nhanh
article = """DÁN BÀI TIN NGẮN VÀO ĐÂY"""
inputs = tokenizer(article, max_length=1024, truncation=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=160, num_beams=4,
                              repetition_penalty=2.5, no_repeat_ngram_size=3,
                              early_stopping=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Nhập 3 bài; URL chỉ lưu nguồn, không tự tải trang.
test_articles = {
 "Bài ngắn": {"url":"https://vnexpress.net/facebook-hoat-dong-chap-chon-5040127.html", "text":""},
 "Bài vừa": {"url":"https://vnexpress.net/siet-ban-hang-online-de-ngan-nhung-vu-nhu-huan-hoa-hong-5114479.html", "text":""},
 "Bài dài": {"url":"https://vnexpress.net/gioi-cong-nghe-chia-phe-vi-ai-noi-loan-5121529.html", "text":""}
}
for name, item in test_articles.items():
    if item["text"].strip():
        print(name, len(tokenizer.encode(item["text"], add_special_tokens=True)), "token")

CHUNK_LIMIT = 800
def split_chunks(text):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], []
    for paragraph in paragraphs:
        candidate = "\n\n".join(current + [paragraph])
        if len(tokenizer.encode(candidate, add_special_tokens=True)) <= CHUNK_LIMIT:
            current.append(paragraph)
        else:
            if current:
                chunks.append("\n\n".join(current))
            current = [paragraph]
    if current:
        chunks.append("\n\n".join(current))
    return chunks

chunks_by_article = {name: split_chunks(item["text"]) for name,item in test_articles.items() if item["text"].strip()}
summaries_by_article = {}
for name, chunks in chunks_by_article.items():
    summaries = []
    for chunk in chunks:
        batch = tokenizer(chunk, max_length=1024, truncation=True, return_tensors="pt")
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.no_grad():
            result = model.generate(**batch, max_new_tokens=160, num_beams=4,
                                    repetition_penalty=2.5, no_repeat_ngram_size=3,
                                    early_stopping=True)
        summaries.append(tokenizer.decode(result[0], skip_special_tokens=True))
    summaries_by_article[name] = summaries
    print("\n", name, "\n", "\n".join(summaries))

final_summaries = {}
for name, parts in summaries_by_article.items():
    text = "\n".join(parts)
    batch = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")
    batch = {k:v.to(device) for k,v in batch.items()}
    with torch.no_grad():
        result = model.generate(**batch, max_new_tokens=160, num_beams=4,
                                repetition_penalty=2.5, no_repeat_ngram_size=3,
                                early_stopping=True)
    final_summaries[name] = tokenizer.decode(result[0], skip_special_tokens=True)
    print("\nBẢN TÓM TẮT:", name, "\n", final_summaries[name])